In [ ]:
import os

# Set these before TensorFlow is imported by any child process.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=0"
os.environ["TF_DISABLE_XLA"] = "1"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTHONHASHSEED"] = "42"

from pathlib import Path
import hashlib
import json
import shutil
import subprocess
import sys
import time

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
CODE_ROOT = WORK_ROOT / "cxr-dg-code"
OUTPUT_ROOT = WORK_ROOT / "dg_suite_final_v2"
RUNNER_LOG_ROOT = OUTPUT_ROOT / "runner_logs"
PATCHED_MANIFEST_ROOT = WORK_ROOT / "patched_manifests"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUNNER_LOG_ROOT.mkdir(parents=True, exist_ok=True)
PATCHED_MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)

assert INPUT_ROOT.is_dir(), "Kaggle input root is unavailable"
print("Python:", sys.version)
print("Input root:", INPUT_ROOT)
print("Output root:", OUTPUT_ROOT)

# Kaggle normally provides these packages; fail before training if the image is incomplete.
for package_name in ("pandas", "sklearn", "PIL", "matplotlib"):
    try:
        __import__(package_name)
    except ImportError as exc:
        raise RuntimeError("Missing required Kaggle package: " + package_name) from exc

try:
    import tensorflow as tf
    print("TensorFlow:", tf.__version__)
    print("Visible GPUs:", tf.config.list_physical_devices("GPU"))
    for device in tf.config.list_physical_devices("GPU"):
        try:
            tf.config.experimental.set_memory_growth(device, True)
        except RuntimeError:
            pass
except Exception as exc:
    raise RuntimeError("TensorFlow must be available in the Kaggle Python image before training.") from exc

In [ ]:
import re
from collections import Counter, defaultdict

SPECIAL_INPUT_NAMES = {
    "cxr_pipeline.py",
    "DG_Final_Execution.py",
    "DG_EXECUTION_RUNBOOK.md",
    "DATASET_MANIFEST.csv",
    "EPIC_FINAL_MANIFEST.csv",
}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".dcm"}

def norm_path(value):
    return str(value or "").replace("\\", "/").strip("/").lower()

def anchor_from_path(value):
    normalized = norm_path(value)
    patterns = (
        r"(images_\d{3}/images/[^/]+)$",
        r"((?:train|valid|test)/patient[^/]+/study[^/]+/[^/]+)$",
        r"((?:training|testing)/[^/]+/[^/]+)$",
    )
    for pattern in patterns:
        match = re.search(pattern, normalized, flags=re.IGNORECASE)
        if match:
            return match.group(1).lower()
    return None

def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

named_inputs = defaultdict(list)
for path in INPUT_ROOT.rglob("*"):
    if path.is_file() and path.name in SPECIAL_INPUT_NAMES:
        named_inputs[path.name].append(path)

def choose_input_file(filename, required_parent_token):
    candidates = sorted(named_inputs.get(filename, []), key=lambda p: (len(str(p)), str(p)))
    if not candidates:
        raise FileNotFoundError("Missing Kaggle input file: " + filename)
    preferred = [
        p for p in candidates
        if required_parent_token.lower() in str(p.parent).lower()
        or required_parent_token.lower() in str(p).lower()
    ]
    chosen = sorted(preferred or candidates, key=lambda p: (len(str(p)), str(p)))[0]
    print("Using", filename, "from", chosen)
    return chosen

PIPELINE_INPUT = choose_input_file("cxr_pipeline.py", "cxr-dg-code")
RUNNER_INPUT = choose_input_file("DG_Final_Execution.py", "cxr-dg-code")
RUNBOOK_INPUT = choose_input_file("DG_EXECUTION_RUNBOOK.md", "cxr-dg-code")

if CODE_ROOT.exists():
    shutil.rmtree(CODE_ROOT)
CODE_ROOT.mkdir(parents=True, exist_ok=True)
for src in (PIPELINE_INPUT, RUNNER_INPUT, RUNBOOK_INPUT):
    shutil.copy2(src, CODE_ROOT / src.name)

pipeline_text = (CODE_ROOT / "cxr_pipeline.py").read_text(encoding="utf-8")
if "1.0.0-dg-2source" not in pipeline_text or "VinDr" in pipeline_text or "VinBigData" in pipeline_text:
    raise RuntimeError("Selected code is not the cleaned 1.0.0 two-source pipeline. Upload the corrected cxr-dg-code folder.")
runner_text = (CODE_ROOT / "DG_Final_Execution.py").read_text(encoding="utf-8")
if "VinDr" in runner_text or "VinBigData" in runner_text or "dg-3source" in runner_text:
    raise RuntimeError("Selected DG runner still contains removed VinDr/VinBigData protocol code.")

# py_compile catches indentation, missing delimiters, and accidental upload corruption.
compile_targets = [CODE_ROOT / "cxr_pipeline.py", CODE_ROOT / "DG_Final_Execution.py"]
compile_result = subprocess.run(
    [sys.executable, "-m", "py_compile", *map(str, compile_targets)],
    cwd=str(CODE_ROOT),
    text=True,
    capture_output=True,
)
if compile_result.returncode != 0:
    print(compile_result.stdout)
    print(compile_result.stderr)
    raise RuntimeError("Static compilation failed; no GPU training was started.")
print("Selected code and static compilation passed.")

# Download/cache ImageNet weights once. Child training processes then reuse the
# same local Keras cache instead of discovering an internet problem later.
try:
    with tf.device("/CPU:0"):
        weight_probe = tf.keras.applications.MobileNetV3Large(
            include_top=False,
            weights="imagenet",
            input_shape=(224, 224, 3),
            include_preprocessing=True,
        )
    del weight_probe
    tf.keras.backend.clear_session()
    print("MobileNetV3Large ImageNet weights are cached and ready.")
except Exception as exc:
    raise RuntimeError(
        "Could not load ImageNet weights. Enable Kaggle Internet for the first run "
        "or attach the Keras weight cache."
    ) from exc

In [ ]:
# NIH uses images_###/images/file; CheXpert uses split/patient/study/view;
# Epic uses Training|Testing/class/file. Filename-only remapping is prohibited.

import csv
# Build the image index after code selection; this avoids delaying Cell 2.
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".dcm"}
anchor_index = {}
ambiguous_anchors = set()
epic_root_scores = Counter()
for path in INPUT_ROOT.rglob("*"):
    if not path.is_file() or path.suffix.lower() not in IMAGE_EXTENSIONS:
        continue
    anchor = anchor_from_path(path)
    if anchor is None:
        continue
    if anchor in anchor_index and anchor_index[anchor] != path:
        ambiguous_anchors.add(anchor)
    else:
        anchor_index[anchor] = path
    parts = list(path.parts)
    for index, part in enumerate(parts):
        if part.lower() in {"training", "testing"} and index > 0:
            epic_root_scores[Path(*parts[:index])] += 1
            break
for anchor in ambiguous_anchors:
    anchor_index.pop(anchor, None)
print("Indexed images:", len(anchor_index), "ambiguous:", len(ambiguous_anchors))

def find_manifest(filename):
    candidates = sorted(named_inputs.get(filename, []), key=lambda p: (len(str(p)), str(p)))
    if not candidates:
        raise FileNotFoundError("Missing required manifest: " + filename)
    preferred = [p for p in candidates if "audited-manifests" in str(p).lower()]
    return sorted(preferred or candidates, key=lambda p: (len(str(p)), str(p)))[0]

SOURCE_INPUT = find_manifest("DATASET_MANIFEST.csv")
EPIC_INPUT = find_manifest("EPIC_FINAL_MANIFEST.csv")
print("Source manifest:", SOURCE_INPUT)
print("Epic manifest:", EPIC_INPUT)

if not anchor_index:
    raise RuntimeError("No recognizable NIH, CheXpert, or Epic image paths were indexed.")

def resolve_image(raw_value):
    anchor = anchor_from_path(raw_value)
    return anchor_index.get(anchor) if anchor else None

def should_patch_source(row):
    return (
        row.get("dataset") in {"NIH ChestX-ray14", "CheXpert"}
        and row.get("harmonization_status", "").lower() == "included"
        and row.get("label", "") in {"0", "1"}
    )

def should_patch_epic(row):
    return (
        row.get("inclusion_status", "").lower() == "included"
        and row.get("external_evaluation_eligible", "").lower() in {"true", "1"}
    )

def patch_manifest(input_path, output_path, predicate):
    unresolved = []
    patched_count = 0
    total = 0
    with open(input_path, "r", encoding="utf-8-sig", newline="") as source_handle:
        reader = csv.DictReader(source_handle)
        fieldnames = reader.fieldnames
        if not fieldnames:
            raise RuntimeError("Manifest has no header: " + str(input_path))
        with open(output_path, "w", encoding="utf-8", newline="") as output_handle:
            writer = csv.DictWriter(output_handle, fieldnames=fieldnames)
            writer.writeheader()
            for row in reader:
                total += 1
                if predicate(row):
                    resolved = resolve_image(row.get("filepath") or row.get("filename"))
                    if resolved is None:
                        unresolved.append(
                            (row.get("sample_id", ""), row.get("filepath", ""), row.get("filename", ""))
                        )
                    else:
                        row["filepath"] = str(resolved)
                        patched_count += 1
                writer.writerow(row)
    if unresolved:
        preview = "\n".join(map(str, unresolved[:5]))
        raise RuntimeError(
            "Unresolved or ambiguous included records: %d\n%s" % (len(unresolved), preview)
        )
    print(output_path.name, "rows:", total, "patched:", patched_count)
    return output_path

PATCHED_SOURCE = patch_manifest(
    SOURCE_INPUT,
    PATCHED_MANIFEST_ROOT / "DATASET_MANIFEST.csv",
    should_patch_source,
)
PATCHED_EPIC = patch_manifest(
    EPIC_INPUT,
    PATCHED_MANIFEST_ROOT / "EPIC_FINAL_MANIFEST.csv",
    should_patch_epic,
)
SOURCE_SHA256 = file_sha256(PATCHED_SOURCE)
EPIC_SHA256 = file_sha256(PATCHED_EPIC)
print("Patched source SHA-256:", SOURCE_SHA256)
print("Patched Epic SHA-256:", EPIC_SHA256)

In [ ]:
# DG_Final_Execution.py uses Epic root / (Training|Testing)/class/file.
# Infer the root from actual mounted paths instead of hard-coding a dataset slug.

epic_candidates = [
    (score, root)
    for root, score in epic_root_scores.items()
    if (root / "Training").is_dir() or (root / "Testing").is_dir()
]
if not epic_candidates:
    raise RuntimeError("Could not infer an Epic root containing Training/Testing.")
EPIC_ROOT = max(
    epic_candidates,
    key=lambda item: (
        item[0],
        int("epic" in str(item[1]).lower() or "chest-x-ray" in str(item[1]).lower()),
        len(str(item[1])),
    ),
)[1]
print("Epic root:", EPIC_ROOT)

PIPELINE_VERSION = "1.0.0-dg-2source"
SEED = 42
BATCH_SIZE = 32
STEPS_PER_EPOCH = 120
EPOCHS = 3
PATIENCE = 2
LEARNING_RATE = 1e-3

COMMON_ARGS = [
    "--pipeline-version", PIPELINE_VERSION,
    "--mode", "KAGGLE",
    "--seed", str(SEED),
    "--input-root", str(INPUT_ROOT),
    "--output-root", str(OUTPUT_ROOT),
    "--manifest", str(PATCHED_SOURCE),
    "--manifest-sha256", SOURCE_SHA256,
    "--epic-manifest", str(PATCHED_EPIC),
    "--epic-root", str(EPIC_ROOT),
    "--batch-size", str(BATCH_SIZE),
    "--steps-per-epoch", str(STEPS_PER_EPOCH),
    "--epochs", str(EPOCHS),
    "--patience", str(PATIENCE),
    "--learning-rate", str(LEARNING_RATE),
    "--required-run-set", "minimum",
]

def safe_slug(value):
    return "".join(ch.lower() if ch.isalnum() else "_" for ch in str(value)).strip("_")

def run_phase(phase, target="Epic Chittagong", method="A", extra_args=None):
    command = [
        sys.executable,
        str(CODE_ROOT / "DG_Final_Execution.py"),
        "--phase", phase,
        "--target", target,
        "--method", method,
        *COMMON_ARGS,
        *(extra_args or []),
    ]
    log_name = "%s_%s_%s_S%s.log" % (phase, safe_slug(target), method, SEED)
    log_path = RUNNER_LOG_ROOT / log_name
    print("\n>>>", " ".join(command))
    environment = os.environ.copy()
    environment["PYTHONUNBUFFERED"] = "1"
    started = time.time()
    with open(log_path, "w", encoding="utf-8") as log_handle:
        process = subprocess.Popen(
            command,
            cwd=str(CODE_ROOT),
            env=environment,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            log_handle.write(line)
        return_code = process.wait()
    elapsed = time.time() - started
    if return_code != 0:
        tail = log_path.read_text(encoding="utf-8", errors="replace")[-4000:]
        raise RuntimeError(
            "Phase failed with exit code %d after %.1f seconds. Log: %s\n%s"
            % (return_code, elapsed, log_path, tail)
        )
    print("<<< completed in %.1f minutes; log: %s" % (elapsed / 60.0, log_path))
    return log_path

In [ ]:
TARGETS = ["Epic Chittagong", "NIH ChestX-ray14", "CheXpert"]
for target in TARGETS:
    run_phase("audit", target=target, method="A")

audit_files = sorted((OUTPUT_ROOT / "reports").glob("DG_AUDIT_*.json"))
if len(audit_files) < 3:
    raise RuntimeError("Expected audit reports for all three targets.")
for audit_path in audit_files:
    audit = json.loads(audit_path.read_text(encoding="utf-8"))
    if audit.get("status") != "PASS":
        raise RuntimeError("Audit failed: " + str(audit_path) + "\n" + json.dumps(audit, indent=2))
    print(
        audit.get("target_domain"),
        "records=", audit.get("records"),
        "missing=", audit.get("audit", {}).get("missing_files"),
        "coverage=", audit.get("audit", {}).get("coverage_issues"),
    )
print("All target audits passed. Training may begin.")

In [ ]:
# All comparison models for a target are trained before any target evaluation.
RUN_MATRIX = {
    "Epic Chittagong": ["A", "B", "C", "D"],
    "NIH ChestX-ray14": ["A", "D"],
    "CheXpert": ["A", "D"],
}

for target, methods in RUN_MATRIX.items():
    print("\n######## TARGET:", target, "########")
    for method in methods:
        run_phase("smoke", target=target, method=method)
    if any(method in {"C", "D"} for method in methods):
        # Lock the predeclared source-only lambda=0.1; no target labels are read.
        run_phase("lambda", target=target, method="C")
    for method in methods:
        run_phase("train", target=target, method=method)
    for method in methods:
        run_phase("evaluate", target=target, method=method)

run_phase("report", target="Epic Chittagong", method="A")
print("Eight-run matrix complete and aggregate gate written.")

In [ ]:
import math
import pandas as pd

evaluation_files = sorted((OUTPUT_ROOT / "logs").rglob("target_evaluation_*.json"))
if len(evaluation_files) != 8:
    raise RuntimeError(
        "Expected exactly 8 evaluation JSON files, found %d. Re-run the failed phase; "
        "completed phases are resumable." % len(evaluation_files)
    )

evaluation_rows = []
for path in evaluation_files:
    payload = json.loads(path.read_text(encoding="utf-8"))
    pooled = payload.get("target_metrics", {}).get("pooled", {})
    evaluation_rows.append({
        "experiment_id": payload.get("experiment_id"),
        "target": payload.get("target_domain"),
        "method": payload.get("method"),
        "seed": payload.get("seed"),
        "n": pooled.get("n"),
        "accuracy": pooled.get("accuracy"),
        "balanced_accuracy": pooled.get("balanced_accuracy"),
        "precision": pooled.get("precision"),
        "recall": pooled.get("recall"),
        "specificity": pooled.get("specificity"),
        "f1": pooled.get("f1"),
        "roc_auc": pooled.get("roc_auc"),
        "pr_auc": pooled.get("pr_auc"),
        "threshold": payload.get("threshold"),
        "evaluation_path": str(path),
        "prediction_path": payload.get("prediction_path"),
        "bootstrap_path": payload.get("bootstrap_path"),
        "calibration_path": payload.get("calibration_path"),
    })

results_df = pd.DataFrame(evaluation_rows)
results_df["roc_auc_sort"] = pd.to_numeric(results_df["roc_auc"], errors="coerce").fillna(-math.inf)
epic_df = results_df[results_df["target"] == "Epic Chittagong"].sort_values(
    ["roc_auc_sort", "f1"], ascending=False
)
if epic_df.empty:
    raise RuntimeError("No Epic evaluation was produced.")
best_row = epic_df.iloc[0].to_dict()
BEST_EXPERIMENT_ID = str(best_row["experiment_id"])
BEST_METHOD = str(best_row["method"])
BEST_EVALUATION_PATH = Path(str(best_row["evaluation_path"]))
BEST_LOCK_PATH = OUTPUT_ROOT / "models" / BEST_EXPERIMENT_ID / "MODEL_LOCK.json"
print("Best Epic method by ROC-AUC:", BEST_METHOD)
print("Best evaluation:", BEST_EVALUATION_PATH)
print(results_df.drop(columns=["roc_auc_sort"]).to_string(index=False))

In [ ]:
run_phase(
    "post",
    target="Epic Chittagong",
    method=BEST_METHOD,
    extra_args=[
        "--experiment-id", BEST_EXPERIMENT_ID,
        "--lock-path", str(BEST_LOCK_PATH),
        "--evaluation-path", str(BEST_EVALUATION_PATH),
    ],
)
print("Post-analysis complete. Grad-CAM and TFLite artifacts are under the best experiment directory.")

In [ ]:
# Cell 9 - generate final figures, confusion matrices, and result tables
import json
import math
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    auc,
    precision_recall_curve,
    roc_curve,
)


FIGURE_ROOT = OUTPUT_ROOT / "figures"
TABLE_ROOT = OUTPUT_ROOT / "tables"
CONFUSION_ROOT = FIGURE_ROOT / "confusion_matrices"

FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
TABLE_ROOT.mkdir(parents=True, exist_ok=True)
CONFUSION_ROOT.mkdir(parents=True, exist_ok=True)


# Save the main experiment results table.
results_df.drop(columns=["roc_auc_sort"], errors="ignore").to_csv(
    TABLE_ROOT / "final_results.csv",
    index=False,
)


# Create the main metric-comparison figure.
metric_names = ["accuracy", "balanced_accuracy", "f1", "roc_auc"]
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

for axis, metric in zip(axes.flat, metric_names):
    plot_df = results_df.copy()
    plot_df[metric] = pd.to_numeric(plot_df[metric], errors="coerce")

    for target in TARGETS:
        subset = plot_df[plot_df["target"] == target]
        if not subset.empty:
            axis.plot(
                subset["method"],
                subset[metric],
                marker="o",
                linewidth=2,
                label=target,
            )

    axis.set_title(metric.replace("_", " ").title())
    axis.set_xlabel("Method")
    axis.set_ylim(0, 1.05)
    axis.grid(alpha=0.3)
    axis.legend(fontsize=8)

fig.suptitle("Two-source DG target metrics", fontsize=15)
fig.tight_layout()
fig.savefig(
    FIGURE_ROOT / "metric_comparison.png",
    dpi=180,
    bbox_inches="tight",
)
plt.close(fig)


# Create ROC and Precision-Recall curves from real prediction files.
roc_fig, roc_axis = plt.subplots(figsize=(8, 7))
pr_fig, pr_axis = plt.subplots(figsize=(8, 7))

for row in evaluation_rows:
    prediction_path = Path(str(row["prediction_path"]))
    predictions = pd.read_csv(prediction_path)

    y_true = predictions["true_label"].astype(int).to_numpy()
    y_prob = predictions["prob"].astype(float).to_numpy()
    label = "%s-%s" % (row["target"], row["method"])

    if np.unique(y_true).size == 2:
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        roc_axis.plot(
            fpr,
            tpr,
            linewidth=1.5,
            label="%s (%.3f)" % (label, auc(fpr, tpr)),
        )

        precision, recall, _ = precision_recall_curve(y_true, y_prob)
        pr_axis.plot(
            recall,
            precision,
            linewidth=1.5,
            label=label,
        )

roc_axis.plot([0, 1], [0, 1], "k--", linewidth=1)
roc_axis.set(
    xlabel="False-positive rate",
    ylabel="True-positive rate",
    title="ROC curves",
)
roc_axis.grid(alpha=0.3)
roc_axis.legend(fontsize=7, loc="lower right")
roc_fig.tight_layout()
roc_fig.savefig(
    FIGURE_ROOT / "roc_curves.png",
    dpi=180,
    bbox_inches="tight",
)
plt.close(roc_fig)

pr_axis.set(
    xlabel="Recall",
    ylabel="Precision",
    title="Precision-Recall curves",
)
pr_axis.grid(alpha=0.3)
pr_axis.legend(fontsize=7, loc="lower left")
pr_fig.tight_layout()
pr_fig.savefig(
    FIGURE_ROOT / "pr_curves.png",
    dpi=180,
    bbox_inches="tight",
)
plt.close(pr_fig)


# Generate one confusion matrix for every completed experiment.
for row in evaluation_rows:
    predictions = pd.read_csv(Path(str(row["prediction_path"])))
    threshold = float(row["threshold"])

    y_true = predictions["true_label"].astype(int).to_numpy()
    y_prob = predictions["prob"].astype(float).to_numpy()
    y_pred = (y_prob >= threshold).astype(int)

    fig, axis = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_predictions(
        y_true,
        y_pred,
        labels=[0, 1],
        display_labels=["Normal", "Pneumonia"],
        cmap="Blues",
        colorbar=False,
        ax=axis,
    )
    axis.set_title(
        "%s - method %s" % (row["target"], row["method"])
    )
    fig.tight_layout()
    fig.savefig(
        CONFUSION_ROOT / ("%s.png" % safe_slug(row["experiment_id"])),
        dpi=180,
        bbox_inches="tight",
    )
    plt.close(fig)


# Build a montage from the real Grad-CAM overlay PNG files.
gradcam_files = sorted(
    (OUTPUT_ROOT / "explainability" / BEST_EXPERIMENT_ID).glob("gradcam_*.png")
)

if gradcam_files:
    columns = min(4, len(gradcam_files))
    rows_count = int(math.ceil(len(gradcam_files) / columns))

    fig, axes = plt.subplots(
        rows_count,
        columns,
        figsize=(4 * columns, 3.5 * rows_count),
    )
    axes = np.atleast_1d(axes).reshape(rows_count, columns)

    for axis in axes.flat:
        axis.axis("off")

    for axis, image_path in zip(axes.flat, gradcam_files):
        with Image.open(image_path) as gradcam_image:
            axis.imshow(gradcam_image.copy())
        axis.set_title(image_path.stem)
        axis.axis("off")

    fig.suptitle("Real Grad-CAM overlays - best Epic model")
    fig.tight_layout()
    fig.savefig(
        FIGURE_ROOT / "gradcam_montage.png",
        dpi=180,
        bbox_inches="tight",
    )
    plt.close(fig)
else:
    raise RuntimeError("No Grad-CAM PNG artifacts were generated.")


# Flatten bootstrap confidence intervals and calibration results.
bootstrap_rows = []
calibration_rows = []

for row in evaluation_rows:
    bootstrap = json.loads(
        Path(str(row["bootstrap_path"])).read_text(encoding="utf-8")
    )
    calibration = json.loads(
        Path(str(row["calibration_path"])).read_text(encoding="utf-8")
    )

    for metric, interval in bootstrap.get("intervals", {}).items():
        bootstrap_rows.append(
            {
                "experiment_id": row["experiment_id"],
                "target": row["target"],
                "method": row["method"],
                "metric": metric,
                "lower_95": interval.get("lower_95"),
                "upper_95": interval.get("upper_95"),
                "iterations_completed": bootstrap.get("iterations_completed"),
            }
        )

    calibration_rows.append(
        {
            "experiment_id": row["experiment_id"],
            "target": row["target"],
            "method": row["method"],
            "ece": calibration.get("uncalibrated", {}).get("ece"),
            "brier_score": calibration.get("uncalibrated", {}).get(
                "brier_score"
            ),
        }
    )

pd.DataFrame(bootstrap_rows).to_csv(
    TABLE_ROOT / "bootstrap_confidence_intervals.csv",
    index=False,
)
pd.DataFrame(calibration_rows).to_csv(
    TABLE_ROOT / "calibration_results.csv",
    index=False,
)


# Collect real TFLite size, latency, and target-performance results.
tflite_reports = []

for report_path in sorted(
    (OUTPUT_ROOT / "quantization").rglob("fp32_vs_int8.json")
):
    payload = json.loads(report_path.read_text(encoding="utf-8"))
    tflite_reports.append(
        {
            "experiment_id": payload.get("experiment_id"),
            "target": payload.get("target_domain"),
            "method": payload.get("method"),
            "quantization_type": payload.get("quantization_type"),
            "fp32_size_bytes": payload.get("fp32_size_bytes"),
            "quantized_size_bytes": payload.get("quantized_size_bytes"),
            "compression_ratio": payload.get("compression_ratio"),
            "tflite_median_ms": payload.get("latency", {}).get(
                "tflite_median_ms"
            ),
            "tflite_metrics": payload.get("target_metrics"),
        }
    )

pd.DataFrame(tflite_reports).to_json(
    TABLE_ROOT / "tflite_efficiency_results.json",
    orient="records",
    indent=2,
)


print(
    "Figures:",
    sorted(
        str(path.relative_to(FIGURE_ROOT))
        for path in FIGURE_ROOT.rglob("*")
        if path.is_file()
    ),
)
print(
    "Tables:",
    sorted(
        str(path.relative_to(TABLE_ROOT))
        for path in TABLE_ROOT.rglob("*")
        if path.is_file()
    ),
)

In [ ]:
protocol = {
    "pipeline_version": PIPELINE_VERSION,
    "sources": ["NIH ChestX-ray14", "CheXpert"],
    "external_target": "Epic Chittagong",
    "run_matrix": RUN_MATRIX,
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "steps_per_epoch": STEPS_PER_EPOCH,
    "epochs_requested": EPOCHS,
    "early_stopping_patience": PATIENCE,
    "learning_rate": LEARNING_RATE,
    "patched_source_manifest": str(PATCHED_SOURCE),
    "patched_source_manifest_sha256": SOURCE_SHA256,
    "patched_epic_manifest": str(PATCHED_EPIC),
    "patched_epic_manifest_sha256": EPIC_SHA256,
    "epic_root": str(EPIC_ROOT),
    "best_epic_experiment_id": BEST_EXPERIMENT_ID,
    "best_epic_method": BEST_METHOD,
    "best_epic_evaluation_path": str(BEST_EVALUATION_PATH),
    "interpretation_note": (
        "For NIH/CheXpert LODO, only one source domain remains; CORAL therefore has "
        "no source-domain pair and its alignment loss is inactive."
    ),
}
(OUTPUT_ROOT / "protocol_metadata.json").write_text(
    json.dumps(protocol, indent=2, sort_keys=True), encoding="utf-8"
)

required_files = [
    FIGURE_ROOT / "metric_comparison.png",
    FIGURE_ROOT / "roc_curves.png",
    FIGURE_ROOT / "pr_curves.png",
    FIGURE_ROOT / "gradcam_montage.png",
    TABLE_ROOT / "final_results.csv",
    TABLE_ROOT / "bootstrap_confidence_intervals.csv",
    TABLE_ROOT / "calibration_results.csv",
    TABLE_ROOT / "tflite_efficiency_results.json",
]
missing = [str(path) for path in required_files if not path.is_file()]
if missing:
    raise RuntimeError("Final artifact validation failed; missing: " + str(missing))

best_gradcam_summary = OUTPUT_ROOT / "explainability" / BEST_EXPERIMENT_ID / "gradcam_summary.json"
best_tflite_report = OUTPUT_ROOT / "quantization" / BEST_EXPERIMENT_ID / "fp32_vs_int8.json"
for path in (best_gradcam_summary, best_tflite_report):
    if not path.is_file():
        raise RuntimeError("Best-model post-analysis artifact missing: " + str(path))

# Include the exact executed code and patched input manifests alongside metrics for an auditable download.
ARCHIVE_MANIFEST_ROOT = OUTPUT_ROOT / "input_manifests"
ARCHIVE_MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)
for manifest in (PATCHED_SOURCE, PATCHED_EPIC):
    shutil.copy2(manifest, ARCHIVE_MANIFEST_ROOT / manifest.name)
ARCHIVE_CODE_ROOT = OUTPUT_ROOT / "code"
ARCHIVE_CODE_ROOT.mkdir(parents=True, exist_ok=True)
for src in (CODE_ROOT / "cxr_pipeline.py", CODE_ROOT / "DG_Final_Execution.py", CODE_ROOT / "DG_EXECUTION_RUNBOOK.md"):
    shutil.copy2(src, ARCHIVE_CODE_ROOT / src.name)

archive_path = shutil.make_archive(
    str(WORK_ROOT / "dg_suite_final"),
    "zip",
    root_dir=str(OUTPUT_ROOT),
)
print("Validated", len(evaluation_files), "evaluation files.")
print("Best model:", BEST_EXPERIMENT_ID)
print("Download archive:", archive_path)
print("Output directory:", OUTPUT_ROOT)